# Exploratory Data Analysis: `diabetes_HT_2`

Source file: `diabetes___HT_2.csv`

This notebook performs a structured EDA: data loading, structure/dtypes, missing-value analysis, descriptive statistics, distributions, categorical breakdowns, and a correlation heatmap, finishing with a single combined dashboard figure saved as PNG.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=0.8)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

dataset_name = 'diabetes_HT_2'


## 1. Load data

In [2]:
csv_path = r"/mnt/user-data/uploads/diabetes___HT_2.csv"
try:
    df = pd.read_csv(csv_path, low_memory=False)
except UnicodeDecodeError:
    df = pd.read_csv(csv_path, low_memory=False, encoding='latin1')

print("Shape:", df.shape)
df.head()


Shape: (26083, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,57.0,1.0,3,145,233,1,0,150,0,2.3,0,0,1,1
1,64.0,0.0,2,130,250,0,1,187,0,3.5,0,0,2,1
2,52.0,1.0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56.0,0.0,1,120,236,0,1,178,0,0.8,2,0,2,1
4,66.0,0.0,0,120,354,0,1,163,1,0.6,2,0,2,1


## 2. Structure & data types

In [3]:
dtype_counts = df.dtypes.astype(str).value_counts()
print("Column count by dtype:")
print(dtype_counts)
print()
buf_info = []
df.info(buf=None)


Column count by dtype:
int64      11
float64     3
Name: count, dtype: int64

<class 'pandas.DataFrame'>
RangeIndex: 26083 entries, 0 to 26082
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       26083 non-null  float64
 1   sex       26058 non-null  float64
 2   cp        26083 non-null  int64  
 3   trestbps  26083 non-null  int64  
 4   chol      26083 non-null  int64  
 5   fbs       26083 non-null  int64  
 6   restecg   26083 non-null  int64  
 7   thalach   26083 non-null  int64  
 8   exang     26083 non-null  int64  
 9   oldpeak   26083 non-null  float64
 10  slope     26083 non-null  int64  
 11  ca        26083 non-null  int64  
 12  thal      26083 non-null  int64  
 13  target    26083 non-null  int64  
dtypes: float64(3), int64(11)
memory usage: 2.8 MB


## 3. Missing value analysis

In [4]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(f"Columns with missing values: {len(missing_df)} / {df.shape[1]}")
missing_df.head(30)


Columns with missing values: 1 / 14


,missing_count,missing_pct
sex,25,0.1


## 4. Descriptive statistics (numeric columns)

In [5]:
numeric_cols_all = df.select_dtypes(include=np.number).columns.tolist()
desc = df[numeric_cols_all].describe().T
desc.head(30)


,count,mean,std,min,25%,50%,75%,max
age,26083.0,55.661389,15.189768,11.0,44.0,56.0,67.0,98.0
sex,26058.0,0.500000,0.500010,0.0,0.0,0.5,1.0,1.0
cp,26083.0,0.958594,1.023931,0.0,0.0,1.0,2.0,3.0
trestbps,26083.0,131.592992,17.588809,94.0,120.0,130.0,140.0,200.0
chol,26083.0,246.246061,51.643522,126.0,211.0,240.0,275.0,564.0
fbs,26083.0,0.149753,0.356836,0.0,0.0,0.0,0.0,1.0
restecg,26083.0,0.526512,0.525641,0.0,0.0,1.0,1.0,2.0
thalach,26083.0,149.655024,22.858109,71.0,133.0,153.0,166.0,202.0
exang,26083.0,0.326573,0.468969,0.0,0.0,0.0,1.0,1.0
oldpeak,26083.0,1.039512,1.165138,0.0,0.0,0.8,1.6,6.2


## 5. Select representative columns for visualization

Given the potentially large number of columns, we pick the most complete (least missing) numeric columns and the most informative low-cardinality categorical columns for plotting.

In [6]:
id_like = []

# Numeric columns: exclude ID-like, exclude near-constant, rank by completeness then variance
num_candidates = [c for c in numeric_cols_all if c not in id_like]
completeness = df[num_candidates].notna().mean()
nunique = df[num_candidates].nunique()
num_candidates = [c for c in num_candidates if nunique.get(c, 0) > 1]
ranked_numeric = (completeness.loc[num_candidates]).sort_values(ascending=False)
plot_numeric_cols = ranked_numeric.head(12).index.tolist()
print("Numeric columns selected for plotting:")
print(plot_numeric_cols)

# Categorical-like columns: object/string dtype OR numeric with small nunique
obj_cols = df.select_dtypes(exclude=np.number).columns.tolist()
low_card_numeric = [c for c in numeric_cols_all if c not in id_like and 2 <= df[c].nunique() <= 10]
cat_candidates = [c for c in (obj_cols + low_card_numeric) if c not in id_like]
cat_candidates = [c for c in cat_candidates if df[c].nunique() <= 15 and df[c].nunique() >= 2]
plot_cat_cols = cat_candidates[:6]
print("Categorical columns selected for plotting:")
print(plot_cat_cols)


Numeric columns selected for plotting:
['age', 'cp', 'chol', 'trestbps', 'fbs', 'restecg', 'slope', 'thalach', 'exang', 'oldpeak', 'thal', 'ca']
Categorical columns selected for plotting:
['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope']


## 6. Correlation heatmap

In [7]:
corr_cols = ranked_numeric.head(15).index.tolist()
plt.figure(figsize=(9, 7))
if len(corr_cols) >= 2:
    corr = df[corr_cols].corr()
    sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, linewidths=0.3, cbar_kws={'shrink': 0.7})
    plt.title(f'Correlation heatmap ({dataset_name})')
    plt.tight_layout()
else:
    plt.text(0.5, 0.5, 'Not enough numeric columns for correlation', ha='center')
plt.show()


## 7. Distributions of key numeric variables

In [8]:
n = len(plot_numeric_cols)
ncols = 4
nrows = int(np.ceil(n / ncols)) if n else 1
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3.5, nrows*2.8))
axes = np.array(axes).reshape(-1)
for i, c in enumerate(plot_numeric_cols):
    ax = axes[i]
    data = df[c].dropna()
    ax.hist(data, bins=30, color='#4C72B0', edgecolor='white')
    ax.set_title(c, fontsize=9)
for j in range(len(plot_numeric_cols), len(axes)):
    axes[j].axis('off')
fig.suptitle(f'Distributions of key numeric variables ({dataset_name})', y=1.02)
plt.tight_layout()
plt.show()


## 8. Categorical variable breakdowns

In [9]:
n = len(plot_cat_cols)
if n:
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*3))
    axes = np.array(axes).reshape(-1)
    for i, c in enumerate(plot_cat_cols):
        ax = axes[i]
        vc = df[c].value_counts(dropna=True).head(10)
        sns.barplot(x=vc.values, y=vc.index.astype(str), ax=ax, color='#55A868')
        ax.set_title(c, fontsize=9)
        ax.set_xlabel('')
    for j in range(len(plot_cat_cols), len(axes)):
        axes[j].axis('off')
    fig.suptitle(f'Categorical variable breakdowns ({dataset_name})', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No suitable low-cardinality categorical columns found.')


## 9. Combined EDA dashboard (single PNG canvas)

All the small plots above are combined into one figure/canvas and exported as a single PNG image for this dataset.

In [10]:
png_path = r"/mnt/user-data/outputs/diabetes_HT_2_EDA_dashboard.png"

n_num = len(plot_numeric_cols)
n_cat = len(plot_cat_cols)
hist_rows = int(np.ceil(n_num / 4)) if n_num else 1
cat_rows = int(np.ceil(n_cat / 3)) if n_cat else 0

total_rows = 2 + hist_rows + cat_rows  # row0: missing+corr, then hist_rows, then cat_rows
fig = plt.figure(figsize=(20, 5 + hist_rows*3.2 + cat_rows*3.2))
gs = gridspec.GridSpec(1 + hist_rows + cat_rows, 4, figure=fig, hspace=0.6, wspace=0.4)

# Row 0a: missingness (spans 2 cols)
ax_miss = fig.add_subplot(gs[0, 0:2])
if len(missing_df):
    top_missing = missing_df.head(15)
    sns.barplot(x=top_missing['missing_pct'], y=top_missing.index.astype(str), ax=ax_miss, color='#C44E52')
    ax_miss.set_title('Top missing-value columns (%)', fontsize=10)
else:
    ax_miss.text(0.5, 0.5, 'No missing values', ha='center')
    ax_miss.axis('off')

# Row 0b: correlation heatmap (spans 2 cols)
ax_corr = fig.add_subplot(gs[0, 2:4])
if len(corr_cols) >= 2:
    sns.heatmap(df[corr_cols].corr(), cmap='coolwarm', center=0, ax=ax_corr, cbar_kws={'shrink': 0.6})
    ax_corr.set_title('Correlation heatmap', fontsize=10)
    ax_corr.tick_params(labelsize=6)
else:
    ax_corr.text(0.5, 0.5, 'Not enough numeric cols', ha='center')
    ax_corr.axis('off')

# Histogram grid
for i, c in enumerate(plot_numeric_cols):
    r = 1 + i // 4
    cpos = i % 4
    ax = fig.add_subplot(gs[r, cpos])
    ax.hist(df[c].dropna(), bins=25, color='#4C72B0', edgecolor='white')
    ax.set_title(c, fontsize=8)
    ax.tick_params(labelsize=6)

# Categorical grid
for i, c in enumerate(plot_cat_cols):
    r = 1 + hist_rows + i // 3
    cpos_group = i % 3
    ax = fig.add_subplot(gs[r, cpos_group])
    vc = df[c].value_counts(dropna=True).head(8)
    sns.barplot(x=vc.values, y=vc.index.astype(str), ax=ax, color='#55A868')
    ax.set_title(c, fontsize=8)
    ax.tick_params(labelsize=6)
    ax.set_xlabel('')

fig.suptitle(f'Combined EDA Dashboard — diabetes_HT_2', fontsize=16, y=1.01)
fig.savefig(png_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved combined dashboard to:', png_path)


Saved combined dashboard to: /mnt/user-data/outputs/diabetes_HT_2_EDA_dashboard.png


## 10. Summary

This notebook covered: dataset shape and dtypes, missingness, descriptive statistics, univariate distributions of the most complete numeric variables, categorical breakdowns of low-cardinality fields, a correlation heatmap, and a single combined dashboard PNG summarizing all of the above.